In [50]:
# pip install polars
import polars as pl
from pathlib import Path
import re

# ===================== Config =====================
BASE_DIR = Path(r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC")

folder_paths = {
    "credit": "Financial Adjustments (Credit) - Transaction Level Exporter.csv",
    "refund": "Financial Adjustments (Expedia-Waived Refund) - Transaction Level Exporter.csv",
    "one_key_cash": "OneKeyCash Adjustments - Transaction Level Exporter.csv",
    "orc_output": r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC\ORC_Output",
}

# Build input file paths from BASE_DIR + relative file names
paths_in = {
    "credit": BASE_DIR / folder_paths["credit"],
    "refund": BASE_DIR / folder_paths["refund"],
    "one_key_cash": BASE_DIR / folder_paths["one_key_cash"],
}

# Output directory comes DIRECTLY from folder_paths (absolute path)
OUT_DIR = Path(folder_paths["orc_output"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Resolved input paths:")
for k, p in paths_in.items():
    print(f"- {k}: {p}")
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")
print(f"Resolved output dir: {OUT_DIR}")

# ===================== Helpers =====================
def drop_leading_index_col(df: pl.DataFrame) -> pl.DataFrame:
    """Drop the first column if it looks like an exported index."""
    if df.width == 0:
        return df
    first = df.columns[0]
    if re.match(r"(?i)^(unnamed:\s*\d+|index)$", first):
        return df.drop(first)
    try:
        s_int = df.select(pl.col(first)).to_series().cast(pl.Int64)
        n = len(s_int)
        if n > 0:
            if (s_int == pl.Series(range(n))).all() or (s_int == pl.Series(range(1, n+1))).all():
                return df.drop(first)
    except Exception:
        pass
    return df

def read_csv(path: Path, source_name: str) -> pl.DataFrame:
    """Read CSV, drop index-like first column, tag source."""
    df = pl.read_csv(
        path,
        infer_schema_length=10_000,
        try_parse_dates=True,
        null_values=["", "NULL", "NaN", "NA"],
        encoding="utf8-lossy",
    )
    df = drop_leading_index_col(df)
    return df.with_columns(pl.lit(source_name).alias("source"))

def to_float(col: str) -> pl.Expr:
    """Sanitize currency text -> float."""
    return (
        pl.col(col)
        .str.replace_all(r"[^\d\-\.,]", "")
        .str.replace_all(",", "")
        .cast(pl.Float64, strict=False)
        .fill_null(0.0)
    )

# ===================== Load =====================
credit_raw = read_csv(paths_in["credit"], "credit")
refund_raw = read_csv(paths_in["refund"], "refund")
okc_raw    = read_csv(paths_in["one_key_cash"], "one_key_cash")

# ===================== Concat (schema-safe) =====================
dfs_utf8 = [df.with_columns(pl.all().cast(pl.Utf8, strict=False)) for df in (credit_raw, refund_raw, okc_raw)]
ORC_Combined = pl.concat(dfs_utf8, how="diagonal")

# ===================== Numeric cleanup & ORC value =====================
ORC_Combined = (
    ORC_Combined
    .with_columns([
        to_float("Payment Amount (USD)").alias("Payment Amount (USD)"),
        to_float("OneKeyCash (USD)").alias("OneKeyCash (USD)"),
        to_float("Gross Booking Amount (USD)").alias("Gross Booking Amount (USD)"),
    ])
    .with_columns(
        (pl.col("Payment Amount (USD)").abs() + pl.col("OneKeyCash (USD)").abs()).alias("ORC_Value (USD)")
    )
)

# ===================== Parse Transaction End Date -> Date =====================
date_parsed = pl.coalesce([
    pl.col("Transaction End Date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S", strict=False).dt.date(),
    pl.col("Transaction End Date").str.strptime(pl.Datetime, format="%m/%d/%Y %H:%M:%S", strict=False).dt.date(),
    pl.col("Transaction End Date").str.strptime(pl.Date,     format="%Y-%m-%d",            strict=False),
    pl.col("Transaction End Date").str.strptime(pl.Date,     format="%m/%d/%Y",            strict=False),
    pl.col("Transaction End Date").str.strptime(pl.Date,     format="%d/%m/%Y",            strict=False),
    pl.col("Transaction End Date").str.strptime(pl.Date,     format="%m/%d/%y",            strict=False),
    pl.col("Transaction End Date").str.strptime(pl.Date,     format="%d/%m/%y",            strict=False),
])
ORC_Combined = ORC_Combined.with_columns(date_parsed.alias("Transaction End Date"))

# ===================== Ckey1 & Ckey2 (dd/mm/yy) =====================
date_text = pl.col("Transaction End Date").dt.strftime("%d/%m/%y").fill_null("")
ORC_Combined = ORC_Combined.with_columns([
    pl.concat_str([date_text, pl.col("Agent Business Location Name").cast(pl.Utf8).fill_null("")]).alias("Ckey1"),
    pl.concat_str([
        date_text,
        pl.col("Agent Email Id").cast(pl.Utf8).fill_null(""),
        pl.col("Agent Business Location Name").cast(pl.Utf8).fill_null(""),
    ]).alias("Ckey2"),
])

# ===================== Month (yy_MM), WeekNo (ISO), Week_Begin (Monday) =====================
ORC_Combined = ORC_Combined.with_columns([
    pl.col("Transaction End Date").dt.strftime("%y_%m").alias("Month"),
    pl.col("Transaction End Date").dt.strftime("%V").cast(pl.Int16).alias("WeekNo"),
    (pl.col("Transaction End Date")
     - pl.duration(days=pl.col("Transaction End Date").dt.strftime("%u").cast(pl.Int8) - 1)
    ).alias("Week_Begin"),
])

# ===================== Export by Month (CSV + Parquet) =====================
df = ORC_Combined.filter(pl.col("Month").is_not_null() & (pl.col("Month") != ""))

print("Row count by Month:")
print(df.group_by("Month").len().sort("Month"))

months = (
    df.select("Month").unique().sort("Month").get_column("Month").to_list()
)

print("Exporting CSV & Parquet by Month...")
for m in months:
    subdf = df.filter(pl.col("Month") == m)
    csv_path = OUT_DIR / f"OCR_Output_{m}.csv"
    subdf.write_csv(csv_path)
    print(f"Saved CSV: {csv_path} | rows={subdf.height}")

all_parquet_path = OUT_DIR / "orc_all.parquet"
ORC_Combined.rechunk().write_parquet(all_parquet_path,compression="snappy",statistics=True)

print("Done.")
print(
    ORC_Combined.select([
        "source", "Transaction End Date", "Month", "WeekNo", "Week_Begin",
        "Payment Amount (USD)", "OneKeyCash (USD)", "Gross Booking Amount (USD)", "ORC_Value (USD)",
        "Ckey1", "Ckey2"
    ]).head(10)
)


Resolved input paths:
- credit: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC\Financial Adjustments (Credit) - Transaction Level Exporter.csv
- refund: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC\Financial Adjustments (Expedia-Waived Refund) - Transaction Level Exporter.csv
- one_key_cash: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC\OneKeyCash Adjustments - Transaction Level Exporter.csv
Resolved output dir: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\ORC\ORC_Output
Row count by Month:
shape: (10, 2)
┌───────┬───────┐
│ Month ┆ len   │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ 25_01 ┆ 10392 │
│ 25_02 ┆ 10989 │
│ 25_03 ┆ 15515 │
│ 25_04 ┆ 15061 │
│ 25_05 ┆ 15132 │
│ 25_06 ┆ 13895 │
│ 25_07 ┆ 12250 │
│ 25_08 ┆ 14648 │
│ 25_09 ┆ 15684 │
│ 25_10 ┆ 6255  │
└───────┴───────┘


In [51]:
ORC_Combined

Agent Business Location Name,Agent First Name,Agent Last Name,Agent Email Id,Transaction End Date,POS Tpid Name,Traveler User Loyalty Tier Name,Point Of Sale Order Reference Number,Line Of Business,Primary Reason Name,Secondary Reason Name,Agent Notes,Application,Gross Booking Amount (USD),Payment Amount (USD),source,OneKeyCash (USD),ORC_Value (USD),Ckey1,Ckey2,Month,WeekNo,Week_Begin
str,str,str,str,date,str,str,str,str,str,str,str,str,f64,f64,str,f64,f64,str,str,str,i16,date
"""Concentrix (Ho Chi Minh City)""","""Thanh Tien""","""Vo""","""thanhtien.vo@concentrix.com""",2025-10-16,"""Expedia US""","""Blue""","""73259719182226""","""CAR""","""Change of Plans""","""Early Checkout""","""SixtRebook and RefundOld ITIN:…","""VOYAGER""",347.44,-100.0,"""credit""",0.0,100.0,"""16/10/25Concentrix (Ho Chi Min…","""16/10/25thanhtien.vo@concentri…","""25_10""",42,2025-10-13
"""Concentrix (Ho Chi Minh City)""","""Bao Tram""","""Doan""","""baotram.doan@concentrix.com""",2025-10-16,"""Expedia US""","""Silver""","""73272847313828""","""INSURANCE""","""Vendor - No Service Provided""","""Car insurance not honored by v…","""insurance not honored by vendo…","""VOYAGER""",65.0,-65.0,"""credit""",0.0,65.0,"""16/10/25Concentrix (Ho Chi Min…","""16/10/25baotram.doan@concentri…","""25_10""",42,2025-10-13
"""Concentrix (Kolkata)""","""PREETY""","""MARU""","""preety.maru1@concentrix.com""",2025-10-16,"""Expedia US""","""Blue""","""73273778704410""","""CAR""","""Change of Plans""","""Early Checkout""","""Waiver approved by Nathan-Cust…","""VOYAGER""",225.89,-225.89,"""credit""",0.0,225.89,"""16/10/25Concentrix (Kolkata)""","""16/10/25preety.maru1@concentri…","""25_10""",42,2025-10-13
"""Concentrix (Ho Chi Minh City)""","""Van Phuc""","""Nguyen""","""vanphuc.nguyen@concentrix.com""",2025-10-16,"""Expedia US""","""Silver""","""73260717407283""","""CAR""","""Change of Plans""","""Early Checkout""","""RCP""","""VOYAGER""",182.48,-100.0,"""credit""",0.0,100.0,"""16/10/25Concentrix (Ho Chi Min…","""16/10/25vanphuc.nguyen@concent…","""25_10""",42,2025-10-13
"""Concentrix (Cairo)""","""Nada""","""Mohamed""","""nadamohamed.mohamed@concentrix…",2025-10-16,"""Expedia Germany""","""Blue""","""73275262265651""","""CAR""","""Change of Plans""","""Early Checkout""",null,"""VOYAGER""",75.4119,-75.69,"""credit""",0.0,75.69,"""16/10/25Concentrix (Cairo)""","""16/10/25nadamohamed.mohamed@co…","""25_10""",42,2025-10-13
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Concentrix (Ho Chi Minh City)""","""Ngan Ha""","""Bui""",null,2025-01-01,"""Expedia UK""",null,"""72996700000000.0""",null,"""Other""",null,"""Member Adjustment- 72996740709…",null,0.0,0.0,"""one_key_cash""",31.3,31.3,"""01/01/25Concentrix (Ho Chi Min…","""01/01/25Concentrix (Ho Chi Min…","""25_01""",1,2024-12-30
"""Concentrix (Kolkata)""","""SHUBHAJEET""","""CHOUDHARY""",null,2025-01-01,null,null,"""72060500000000.0""",null,"""Goodwill Bonus""",null,"""Extra Rewards- 72060490872964""",null,0.0,0.0,"""one_key_cash""",25.0,25.0,"""01/01/25Concentrix (Kolkata)""","""01/01/25Concentrix (Kolkata)""","""25_01""",1,2024-12-30
"""Concentrix (Ho Chi Minh City)""","""Ba Phuc""","""Tran""",null,2025-01-01,"""Expedia US""",null,"""72961300000000.0""",null,"""Goodwill Bonus""",null,"""Extra Rewards""",null,0.0,0.0,"""one_key_cash""",100.0,100.0,"""01/01/25Concentrix (Ho Chi Min…","""01/01/25Concentrix (Ho Chi Min…","""25_01""",1,2024-12-30
